In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import pickle

import re

import glob

from itertools import combinations, chain

## Import pseudopazs of excel as a list of series

In [ ]:
folder_path = '../raw_data/pickles/pps/*-pp-uniq.pickle'

path_list = sorted(glob.glob(folder_path))

sers = []
for path in path_list:
    with open(path, 'rb') as file:
        ser = pickle.load(file)

        ser = ser.reset_index(drop=True) # reset the index?

    sers.append(ser)

## Make a list of the series uniqued and sorted

In [ ]:
sers[1]

## Convert series to sets

In [ ]:
# Convert to sets
sets = []
for ser in sers:
    sets.append(set(ser))


set_names = []
for path in path_list:
    set_names.append(path[path.rfind('/')+1 : path.rfind('-pp-uniq.pickle')])

# Get all unique items
all_items = set().union(*sets)
total_unique = len(all_items)

## Compute the intersection between sets

In [ ]:
# Calculate all intersections
intersections = []

# For each possible combination of sets
for r in range(1, len(sets) + 1):
    for combo in combinations(range(len(sets)), r):
        # Get items in exactly this combination
        in_sets = [sets[i] for i in combo]
        out_sets = [sets[i] for i in range(len(sets)) if i not in combo]

        # Items in all "in" sets
        items = set.intersection(*in_sets) if in_sets else set()

        # But not in any "out" sets
        for out_set in out_sets:
            items = items - out_set

        if len(items) > 0:
            intersections.append({
                'sets': combo,
                'count': len(items),
                'percentage': (len(items) / total_unique * 100)
            })

# Sort by count (descending)
intersections.sort(key=lambda x: x['count'], reverse=True)

Stats about the intersections

In [ ]:
for inter in intersections:
    set_list = [set_names[i] for i in inter['sets']]
    if len(set_list) == 1:
        print(f'  Only {set_list[0]}: {inter['count']} items ({inter['percentage']:.1f}%)')
    else:
        print(f'  {' ∩ '.join(set_list)}: {inter['count']} items ({inter['percentage']:.1f}%)')
if len(intersections) > 10:
    print(f'  ... and {len(intersections) - 10} more intersections')

## Plot

In [ ]:
# from pyfonts import load_google_font

# plt.rcParams['font.family'] = 'sans-serif'
# plt.rcParams['font.sans-serif'] = ['Roboto']

In [ ]:
# Create UpSet plot
fig = plt.figure(figsize=(14, 10))

# Create grid: top part for bar chart, bottom for matrix
gs = fig.add_gridspec(2, 1, height_ratios=[2, 1], hspace=0.05)
ax_bars = fig.add_subplot(gs[0])
ax_matrix = fig.add_subplot(gs[1])

# Prepare data for plotting
x_positions = range(len(intersections))
counts = [inter['count'] for inter in intersections]
percentages = [inter['percentage'] for inter in intersections]

# Plot bars with explicit alignment
bars = ax_bars.bar(x_positions, counts, color='#2E86AB', width=0.6, align='center')
ax_bars.set_ylabel("# di pseudop nell'intersezione", fontsize=16, fontweight='bold')
ax_bars.set_xticks([])
ax_bars.set_xlim(-0.5, len(intersections) - 0.5)  # Match matrix xlim for alignment
ax_bars.spines['top'].set_visible(False)
ax_bars.spines['right'].set_visible(False)
ax_bars.spines['bottom'].set_visible(False)
ax_bars.grid(axis='y', alpha=0.3, linestyle='--')

# Add count and percentage labels on bars
for i, (bar, count, pct) in enumerate(zip(bars, counts, percentages)):
    height = bar.get_height()
    ax_bars.text(bar.get_x() + bar.get_width()/2., height,
                f'{count}\n({pct:.1f}%)',
                ha='center', va='bottom', fontsize=12)

# Plot intersection matrix
matrix_data = np.zeros((len(sets), len(intersections)))
for i, inter in enumerate(intersections):
    for set_idx in inter['sets']:
        matrix_data[set_idx, i] = 1

# Draw matrix - first draw all gray dots, then overlay active ones
for row in range(len(sets)):
    for col in range(len(intersections)):
        if matrix_data[row, col] == 1:
            ax_matrix.plot(col, row, 'o', markersize=15, color='#2E86AB', zorder=3)
        else:
            # More visible gray dots
            ax_matrix.plot(col, row, 'o', markersize=15, color='lightgray', zorder=1)

# Draw connecting lines for combinations (without redrawing circles)
for col, inter in enumerate(intersections):
    set_indices = list(inter['sets'])
    if len(set_indices) > 1:
        y_coords = set_indices
        x_coords = [col] * len(y_coords)
        ax_matrix.plot(x_coords, y_coords, '-', linewidth=3, color='#2E86AB', zorder=2)

# Configure matrix axes
ax_matrix.set_yticks(range(len(sets)))
ax_matrix.set_yticklabels(set_names, fontsize=11)
ax_matrix.set_xticks([])
ax_matrix.set_xlim(-0.5, len(intersections) - 0.5)  # Match bars xlim for perfect alignment
ax_matrix.set_ylim(-0.5, len(sets) - 0.5)
ax_matrix.spines['top'].set_visible(False)
ax_matrix.spines['right'].set_visible(False)
ax_matrix.spines['bottom'].set_visible(False)
ax_matrix.spines['left'].set_visible(False)
ax_matrix.invert_yaxis()

# Add set size labels on the RIGHT side for better layout
for i, (name, s) in enumerate(zip(set_names, sets)):
    ax_matrix.text(len(intersections) - 0.3, i, f'({len(s)})', ha='left', va='center',
                  fontsize=10, color='gray')

# Overall title
fig.suptitle('Presenza di pseudopaz nei diversi dataset (n_tot = 137)', fontsize=20, fontweight='bold', y=0.98)

# Add legend for set sizes on the right
ax_matrix.text(len(intersections) - 0.4, -0.8, '↓ Set sizes', ha='left', va='center',
              fontsize=9, style='italic', color='gray')

plt.tight_layout()
plt.savefig('../plots/upset_plot-ori.pdf', dpi=300, bbox_inches='tight')

plt.show()